In [42]:
import json
from pathlib import Path
import requests
from bs4 import BeautifulSoup as bs
import hashlib


# json.load() 接受的是已打开的文件对象，不能直接传文件路径
source_path = Path("./data/v2/source_candidates.json")
failed_downloads = list()
with source_path.open("r", encoding="utf-8") as f:
    source_candidates = json.load(f)
for source_candidate in source_candidates:
    
    source_url = source_candidate["url"]
    try:
        response = requests.get(source_url, timeout=30)
        response.raise_for_status()
    except requests.exceptions.RequestException as exc:
        failed_info = {
            "id": source_candidate["id"],
            "url": source_url,
            "status": exc.response.status_code if exc.response is not None else None,
            "exception_message": str(exc)
        }
        failed_downloads.append(failed_info)
        continue
    
    content = response.content
    digest = hashlib.sha256(content).hexdigest()
    
    data_path = Path.cwd() / "data" / "v2" / "raw" 
    Path.mkdir(data_path, exist_ok=True)
    
    content_type = response.headers.get("Content-Type", "").lower() # 响应头说明服务器返回的内容类型
    page_title = source_candidate["title"] # 非 HTML 或缺少标题时，使用候选清单中的标题
    if "text/html" in content_type or "application/xhtml+xml" in content_type:
        html = bs(content, "lxml")
        if html.title is not None: # 找到 <title> 标签后，才能提取其中的文字
            page_title = html.title.get_text(strip=True) # 去掉文字首尾的空白
            suffix = ".html"
    if "application/pdf" in content_type:
            suffix = ".pdf"
        
    raw_path = data_path / (source_candidate["id"] + suffix)
    raw_path.write_bytes(content)

    saved_digest = hashlib.sha256(raw_path.read_bytes()).hexdigest() # read_bytes() 返回字节数据，而 sha256 需要字节数据, read_text() 返回的是字符串

    print("HTTP 状态：", response.status_code)
    print("最终 URL：", response.url)
    print("页面标题：", page_title)
    print("下载字节数：", len(response.content))
    print("保存位置：", raw_path.resolve())
    print("SHA-256：", digest)
    print("下载与保存内容一致：", digest == saved_digest)
    
print("失败来源数：", len(failed_downloads))
for failure in failed_downloads:
    print(failure)
    

HTTP 状态： 200
最终 URL： https://www.acr.org/clinical-resources/clinical-tools-and-reference/reporting-and-data-systems/lung-rads
页面标题： ACR Lung CT Screening Reporting & Data System (Lung-RADS®)
下载字节数： 155519
保存位置： /home/wzh/clinical-evidence-rag/data/v2/raw/acr-lungrads-scope.html
SHA-256： b5ca0d54a291f7646723c056d329aca412f4ac08d149d7c96b9632af3fdd3328
下载与保存内容一致： True
HTTP 状态： 200
最终 URL： https://edge.sitecorecloud.io/americancoldf5f-acrorgf92a-productioncb02-3650/media/ACR/Files/RADS/Lung-RADS/Lung-RADS-2022.pdf
页面标题： Lung-RADS v2022 Assessment Categories and Management Recommendations
下载字节数： 665034
保存位置： /home/wzh/clinical-evidence-rag/data/v2/raw/acr-lungrads-2022.pdf
SHA-256： 3eca605a00ab4bf8985ecffd3bd614dcee5980578e309cd5599246394cea61bd
下载与保存内容一致： True
HTTP 状态： 200
最终 URL： https://acsearch.acr.org/docs/69455/Narrative/
页面标题： Preview
下载字节数： 275180
保存位置： /home/wzh/clinical-evidence-rag/data/v2/raw/acr-incidental-2023.html
SHA-256： 6491745ffe05c22c500054949bb209744369de8e873e7bbeb81b

In [ ]:
import os

    
for file in data_path.iterdir():
    if file.is_file():
        extension = file.suffix.lower()
        if extension in (".html", ".htm") and "nci-lung-screening-pdq-hp" in str(file):
            with file.open("r", encoding="utf-8") as f:
                
                html_text = f.read()
                html = bs(html_text, "lxml")
                article = html.find("article")
                h1 = article.find("h1")
                h1_text = h1.get_text(separator=" ", strip=True)
                section_titles = article.find_all("h2")
                if not section_titles:
                    section_titles = article.find_all("h3")
                section_titles_texts = [x.get_text(separator=" ", strip=True) for x in section_titles] # get_text 两个参数的作用
                    
                # print("正文容器存在：", article is not None)
                # print("正文主标题：", h1_text)
                # print("前三个章节：", section_titles[:3])
                
                for section_title in section_titles:
                    section_title_text = section_title.get_text(separator=" ", strip=True)
                    if section_title_text == "Risk Factors":
                        section = section_title.find_parent("section")
                        section_id = section.get("id")
                        paragraph = section.find_all("p")
                        paragraph_texts = [x.get_text(separator=" ", strip=True) for x in paragraph]
                        source = next(
                            (candidate for candidate in source_candidates 
                             if "nci-lung" in candidate["id"]
                            ),
                            None
                        )
                        if source:
                            section_locator = f"{source["url"]}#{section_id}"
                            
                            section_record = {
                                "source_id": source["id"],
                                "section_title": section_title_text,
                                "section_id": section_id,
                                "paragraphs": paragraph_texts,
                                "section_url": section_locator                         
                            }

                        
                        print("章节标题：", section_title_text)
                        print("章节 ID：", section_id)
                        print("段落数：", len(paragraph_texts))
                        print("原文定位：", section_locator)
                        print("首段预览：", paragraph_texts[0][:120])
                
                # print("章节标题：", section_title)
                # print("章节 ID：", section_id)
                # print("段落数：", len(paragraph_texts))
                # print("原文定位：", section_url)
                # print("首段预览：", paragraph_texts[0][:120])

章节标题： Risk Factors
章节 ID： _222
段落数： 2
原文定位： https://www.cancer.gov/types/lung/hp/lung-screening-pdq#_222
首段预览： The most important risk factor for lung cancer (as for many other
cancers) is tobacco use.[ 1 , 2 ]  Cigarette smoking h


In [ ]:
def parse_nci_section(html_path, source_record, target_title):
    extension = os.path.splitext(html_path)[1].lower()
    if not (extension in (".html", ".htm") and os.path.isfile(html_path)): raise TypeError("需要 Html 格式文件")
    if not source_record: raise ValueError("source_record == None")
    with html_path.open("r", encoding="utf-8") as f:
        html_text = f.read()
        html = bs(html_text, "lxml")
        article = html.find("article")
        h1 = article.find("h1")
        h1_text = h1.get_text(separator=" ", strip=True)
        section_titles = article.find_all("h2")
        if not section_titles:
            section_titles = article.find_all("h3")
        section_titles_texts = [x.get_text(separator=" ", strip=True) for x in section_titles] # get_text 两个参数的作用
            
        # print("正文容器存在：", article is not None)
        # print("正文主标题：", h1_text)
        # print("前三个章节：", section_titles[:3])
        
        for section_title in section_titles:
            section_title_text = section_title.get_text(separator=" ", strip=True)
            if section_title_text == target_title:
                section = section_title.find_parent("section")
                section_id = section.get("id")
                paragraph = section.find_all("p")
                paragraph_texts = [x.get_text(separator=" ", strip=True) for x in paragraph]
                if source_record:
                    section_locator = f"{source_record["url"]}#{section_id}"
                    
                    section_card = {
                        "source_id": source_record["id"],
                        "section_title": section_title_text,
                        "section_id": section_id,
                        "paragraphs": paragraph_texts,
                        "section_url": section_locator                         
                    }
                return section_card
        raise ValueError("不存在的章节名")    

In [ ]:

from pathlib import Path
import pypdf
pdf_url = Path("./data/v2/raw/acr-lungrads-2022.pdf")
pdf_reader = pypdf.PdfReader(pdf_url)
acr_pdf_pages = pdf_reader.pages
pdf_pages = list()
for index, page in enumerate(acr_pdf_pages, start=1):
    pdf_page = {
        "source_id": "acr-lungrads-2022",
        "page": index,
        "text": page.extract_text()
    }
    pdf_pages.append(pdf_page)



print("PDF 页数：", len(pdf_reader.pages))
print("页记录数：", len(pdf_pages))



for record in pdf_pages:
    print(f"第 {record['page']} 页，字符数：{len(record['text'])}")
    print(record["text"][:120])

PDF 页数： 2
页记录数： 2
第 1 页，字符数：4322
Lung-
RADS Category Descriptor Findings Management
0
Incomplete
Estimated Population
Prevalence: ~ 1%
Prior chest CT exa
第 2 页，字符数：7770
NOTES
1. Lung-RADSCategory: Each exam should be coded 0-4 based on the nodule with the highest degree of suspicion.
2. L


### PDF 编号注释分段（助手按本次授权实现）

实现位于 `document_parsing.py`：通用编号分段与 Lung-RADS 范围适配分别处理。
下面只读取本地页记录，保存带来源、页码及原文字符位置的注释。
该代码已在独立 Python 进程执行；此处不伪造 Notebook 的执行次数或输出。
这是未经医学逐条复核的中间解析结果，尚未加入检索索引。

In [5]:
from pathlib import Path
import json
from document_parsing import parse_lungrads_notes

# 本步由助手按授权实现；仅读取本地已保存的页记录。
pages_path = Path("data/v2/parsed/acr-lungrads-2022.pages.json")
page_records = json.loads(pages_path.read_text(encoding="utf-8"))
notes_page_record = next(
    record for record in page_records
    if record["source_id"] == "acr-lungrads-2022" and record["page"] == 2
)
notes = parse_lungrads_notes(notes_page_record)

notes_path = Path("data/v2/parsed/acr-lungrads-2022.notes.json")
notes_path.parent.mkdir(parents=True, exist_ok=True)
notes_path.write_text(
    json.dumps(notes, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)

print("注释数量：", len(notes))
print("编号列表：", [item["note_number"] for item in notes])
print("原文切片一致：", all(
    item["text"] == notes_page_record["text"][item["char_start"]:item["char_end"]]
    for item in notes
))
print("保存位置：", notes_path.resolve())


注释数量： 16
编号列表： [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
原文切片一致： True
保存位置： /home/wzh/clinical-evidence-rag/data/v2/parsed/acr-lungrads-2022.notes.json


In [ ]:
# CT v2 数据准备：助手按本次整批授权实现，核心检索与评分由你继续写。
from ct_data_pipeline import build_corpus, build_cases
from validate_ct_data import validate

corpus_summary = build_corpus()
case_summary = build_cases()
data_checks = validate()
print("候选证据：", corpus_summary["unique_candidate_count"])
print("本地检索输入：", corpus_summary["retrieval_ready_count"])
print("合成题目：", case_summary["case_count"])
print("场景家族：", case_summary["family_count"])
print("开发／留出：", case_summary["by_split"])
print("全部数据检查通过：", all(data_checks["checks"].values()))
